# Chapter 9: a micro risk score against the fixed list
14,222 synthetic micro and hygiene results: 26 sample points x 547 days, seed 7. The generator plants 5 hard to clean points (two bright tank outlets and three filler fittings) and a CIP slip in the test window. A logistic regression trains on the first 365 days and ranks the 26 points each morning of the last 182.

**No code tier.** Pivot the first 365 days by sample point and count positives. Rank the points and take the top five as your fixed list. Count what share of the last 182 days' positives fell on that list.

Synthetic data from Kestrel Brewing, an invented plant. Copyright 2026 Ankur Napa. All rights reserved. You may use these files for your own learning; do not republish or redistribute them.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
DATA = "../data/"
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings("ignore")

In [ ]:
m = pd.read_csv(DATA + "ch09_micro.csv")
feats = ["days_since_cip", "cip_short", "asset_age_years", "days_since_known_positive", "hard_fitting"]
print(len(m), "results;", m.positive.sum(), "positives")
print(m.groupby("hard_fitting").positive.agg(["sum", "count", "mean"]))
print(m.groupby("cip_short").positive.mean().mul(100).round(1))

In [ ]:
train, test = m[m.day < 365], m[m.day >= 365].copy()
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(train[feats], train.positive)
test["score"] = model.predict_proba(test[feats])[:, 1]
flags = test.sort_values(["day", "score"], ascending=[True, False], kind="stable").groupby("day").head(5)
found, pos = flags.positive.sum(), test.positive.sum()
fixed = test[test.hard_fitting == 1].positive.sum()
print(f"flags {len(flags)}  found {found}  precision {found / len(flags):.1%}  recall {found / pos:.1%}")
print(f"random 5 a day would find {len(flags) * pos / len(test):.0f}; the fixed list of hard fittings finds {fixed}")

In [ ]:
rank = flags.sample_point.value_counts()
rank.plot.bar(figsize=(9, 3), color="#2f7f72"); plt.ylabel("days in the top five"); plt.title("Which points the score sends the microbiologist to"); plt.show()

## The leakage trap
Count a positive as known on the day it was sampled, not when the plate is read. The model then sees the future.

In [ ]:
leak = m.copy()
last = {}
vals = []
for p, d, y in zip(leak.sample_point, leak.day, leak.positive):
    vals.append(min(90, d - last[p]) if p in last else 90)
    if y: last[p] = d
leak["days_since_known_positive"] = vals
# the leaked feature still lags one day here; shift it to the sampling day itself
leak["days_since_known_positive"] = np.where(leak.positive == 1, 0, leak.days_since_known_positive)
tr, te = leak[leak.day < 365], leak[leak.day >= 365].copy()
te["score"] = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(tr[feats], tr.positive).predict_proba(te[feats])[:, 1]
print("with a leaked feature the score finds", te.sort_values(["day", "score"], ascending=[True, False]).groupby("day").head(5).positive.sum(), "of", te.positive.sum())

**What to look at.** The model adds 4 positives to the 90 the fixed list already finds. The leaked version looks brilliant and is useless at 06:00. Azure Machine Learning designer or a pivot table can build the baseline.